# 03 · Video perturbations

How well each network still detects a collision when the video is degraded:
noisy, flickering, shaky or low-contrast.

Accuracy is the fraction of frames a network still gets right, out of those it
got right on the clean video. Sweeping a perturbation's strength gives an
accuracy curve; `R` is the area under it and `LERS = R / log10(FLOPs)` is `R`
per order of magnitude of compute.

Noise strength is scaled to each network's input size, so RAFT (173k pixels)
and FlyNet (721 columns) take the same total noise power.

Needs `flyvis`, `pytorch3d`, a RAFT checkout and the trained decoders. Run from
the repository root.

In [ ]:
import json, torch, numpy as np
from flynet_robustness import config
from flynet_robustness.evaluation import PERTURBATIONS, sweep_perturbation
from flynet_robustness.models import load_network

device = 'cuda' if torch.cuda.is_available() else 'cpu'
trajectories = json.load(open('data/configs/attack_trajectories.json'))

print('networks     :', config.PAPER_NETWORKS)
print('perturbations:', list(PERTURBATIONS))

## One network, one perturbation

`sweep_perturbation` does the whole measurement in one call: 10 trajectories
(20 for fog), 10 noise draws on each, `R` bootstrapped over trajectories.

In [ ]:
network_id = 'FlyNet'      # any id: 'CTRNN', 'sCTRNN', 'vCNN_5_frames', 'RAFT_12', 'RAFT_12_S', ...
name       = 'Flicker'     # AWN_texture, AWN_frames, Shot_noise, Flicker, Motion_jitter, Fog

net, decoder = load_network(network_id, device=device)
res = sweep_perturbation(net, decoder, network_id, name, trajectories, device=device)
print(f"{network_id} / {name}: R = {res['robustness']:.3f} ± {res['robustness_std']:.3f}"
      f"   LERS = {res['lers']:.3f}   ({res['n_trajectories']} trajectories)")

In [ ]:
from flynet_robustness.plotting import plot_accuracy_curve
acc = res['accuracy']                                   # (magnitudes, trajectories)
plot_accuracy_curve(res['magnitudes'], {network_id: acc.mean(1)},
                    xlabel=config.PERTURBATION_AXIS[name][0],
                    errors={network_id: acc.std(1) / np.sqrt(acc.shape[1])});

## Every network, every perturbation

The full table takes many GPU-hours, so the command line is the practical way:

```bash
python scripts/run_perturbations.py --networks all --results results
```

Same thing below, kept small so you can watch it work — widen `NETWORKS` and
`NAMES` for the real run.

In [ ]:
NETWORKS = config.PAPER_NETWORKS[:2]
NAMES    = ['Flicker', 'Fog']

table = {}
for nid in NETWORKS:
    net, decoder = load_network(nid, device=device)
    for pname in NAMES:
        r = sweep_perturbation(net, decoder, nid, pname, trajectories, device=device)
        table[(nid, pname)] = (r['robustness'], r['robustness_std'])
        print(f"{nid:16s} {pname:14s} R = {r['robustness']:.3f} ± {r['robustness_std']:.3f}"
              f"   LERS = {r['lers']:.3f}", flush=True)
    del net, decoder
    torch.cuda.empty_cache() if device == 'cuda' else None

## Check against the published numbers

`data/reference_results/` holds the `R` values this benchmark produced, so a
rerun can be checked numerically. The noise is redrawn each time, so expect
agreement to within about one `R_std`.

In [ ]:
import csv
for (nid, pname), (R, R_std) in table.items():
    ref = {r['network']: r for r in
           csv.DictReader(open(f'data/reference_results/Environmental_{pname}.csv'))}
    want = float(ref[nid]['R'])
    print(f"{nid:16s} {pname:14s} ours {R:.3f} ± {R_std:.3f}   published {want:.3f}"
          f"   Δ = {R - want:+.3f}")